# 8.1 Graph Optimizations — Apply

## Objective

Put graph optimization theory into practice. You will:

1. Build models with redundant operations and apply ORT optimization levels
2. Demonstrate constant folding, dead code elimination, and identity removal
3. Implement Conv+BN and MatMul+Add operator fusion end-to-end
4. Measure real performance improvements from optimization
5. Use `onnxoptimizer` passes for offline graph cleanup

**Prerequisites:** `pip install onnx onnxruntime onnxoptimizer numpy matplotlib`

**Key formula — memory savings from fusing $k$ element-wise ops on tensor $T$:**

$$\text{Savings} = (k-1) \times 2 \times |T| \times \text{sizeof}(\text{dtype})$$

In [ ]:
# Setup
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper
from onnx.checker import check_model
import onnxruntime as ort
import time
import os
import tempfile

try:
    import onnxoptimizer
    HAS_OPTIMIZER = True
except ImportError:
    HAS_OPTIMIZER = False

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

TMPDIR = tempfile.mkdtemp(prefix='onnx_opt_')

print(f"onnx {onnx.__version__}, onnxruntime {ort.__version__}")
if HAS_OPTIMIZER:
    print(f"onnxoptimizer {onnxoptimizer.__version__}")
print(f"Temp dir: {TMPDIR}")

---
## Exercise 1: Build a Model with Redundant Operations

We construct a model containing **Identity nodes**, **constant subexpressions**, and
**dead branches** — the kind of bloat that real exporters produce.

The computation graph:
```
x → Identity → Add(const_a, const_b) → Mul(x, folded) → Identity → y
                                          ↑ dead: Neg(x) → unused
```

After optimization, only the essential `Mul` (or `Sub`) should remain.

In [ ]:
np.random.seed(42)

a_val = np.array([3.0], dtype=np.float32)
b_val = np.array([5.0], dtype=np.float32)

a_init = numpy_helper.from_array(a_val, name='const_a')
b_init = numpy_helper.from_array(b_val, name='const_b')

nodes = [
    helper.make_node('Identity', ['x'], ['x_id']),          # redundant
    helper.make_node('Add', ['const_a', 'const_b'], ['ab']),# constant-foldable
    helper.make_node('Mul', ['x_id', 'ab'], ['mul_out']),   # becomes Mul(x, 8.0)
    helper.make_node('Identity', ['mul_out'], ['y']),       # redundant
    helper.make_node('Neg', ['x'], ['dead_neg']),           # dead code
    helper.make_node('Exp', ['dead_neg'], ['dead_exp']),    # dead code
]

x_info = helper.make_tensor_value_info('x', TensorProto.FLOAT, [4])
y_info = helper.make_tensor_value_info('y', TensorProto.FLOAT, [4])

graph = helper.make_graph(nodes, 'redundant_graph',
                          inputs=[x_info], outputs=[y_info],
                          initializer=[a_init, b_init])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
check_model(model)

print(f"Original model: {len(model.graph.node)} nodes")
for n in model.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)}) -> {list(n.output)}")
print(f"Initializers: {[i.name for i in model.graph.initializer]}")

# Run it to confirm correctness
sess = ort.InferenceSession(model.SerializeToString())
x_test = np.array([1.0, 2.0, 3.0, 4.0], dtype=np.float32)
result = sess.run(None, {'x': x_test})[0]
expected = x_test * (a_val + b_val)
print(f"\nf([1,2,3,4]) = x * (3+5) = {expected.flatten()}")
print(f"Model output:              {result}")
assert np.allclose(result, expected), "Mismatch!"

---
## Exercise 2: Apply ORT Optimization Levels (DISABLED → BASIC → EXTENDED → ALL)

ORT provides four optimization levels:

| Level | Constant | Passes |
|:---:|:---|:---|
| 0 | `ORT_DISABLE_ALL` | None |
| 1 | `ORT_ENABLE_BASIC` | Constant folding, identity/dropout removal |
| 2 | `ORT_ENABLE_EXTENDED` | + Conv+BN, MatMul+Add fusions |
| 99 | `ORT_ENABLE_ALL` | + Layout transforms, all EP-specific |

We save each optimized model and compare node counts.

In [ ]:
levels = [
    ('DISABLED', ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
    ('BASIC',    ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
    ('EXTENDED', ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
    ('ALL',      ort.GraphOptimizationLevel.ORT_ENABLE_ALL),
]

opt_results = {}

for name, level in levels:
    so = ort.SessionOptions()
    so.graph_optimization_level = level
    outpath = os.path.join(TMPDIR, f'opt_{name}.onnx')
    so.optimized_model_filepath = outpath
    sess = ort.InferenceSession(model.SerializeToString(), so)
    opt_model = onnx.load(outpath)
    opt_results[name] = {
        'node_count': len(opt_model.graph.node),
        'ops': [n.op_type for n in opt_model.graph.node],
        'output': sess.run(None, {'x': x_test})[0],
    }

print(f"{'Level':<12} {'Nodes':>5}  Op Types")
print('-' * 60)
for name, info in opt_results.items():
    print(f"{name:<12} {info['node_count']:>5}  {info['ops']}")
    assert np.allclose(info['output'], expected), f"{name}: output mismatch!"

print(f"\nAll levels produce identical outputs: True")
print(f"Node reduction: {opt_results['DISABLED']['node_count']} -> {opt_results['ALL']['node_count']}")

---
## Exercise 3: Constant Folding Demonstration

Constant folding pre-evaluates subgraphs whose inputs are all known at optimization time.
Given constants $c_1, c_2, c_3$:

$$y = x - \bigl((c_1 + c_2) \times c_3\bigr) \xrightarrow{\text{fold}} y = x - c_{\text{folded}}$$

The folding is iterative (cascading): folding `Add` creates a new constant that makes `Mul` foldable.

In [ ]:
c1 = numpy_helper.from_array(np.array([2.0], dtype=np.float32), 'c1')
c2 = numpy_helper.from_array(np.array([3.0], dtype=np.float32), 'c2')
c3 = numpy_helper.from_array(np.array([4.0], dtype=np.float32), 'c3')

fold_nodes = [
    helper.make_node('Add', ['c1', 'c2'], ['sum12']),     # fold iter 1: 2+3=5
    helper.make_node('Mul', ['sum12', 'c3'], ['prod']),   # fold iter 2: 5*4=20
    helper.make_node('Sub', ['x', 'prod'], ['y']),        # remains: y = x - 20
]

x_info = helper.make_tensor_value_info('x', TensorProto.FLOAT, [1])
y_info = helper.make_tensor_value_info('y', TensorProto.FLOAT, [1])
graph = helper.make_graph(fold_nodes, 'cascading_fold',
                          inputs=[x_info], outputs=[y_info],
                          initializer=[c1, c2, c3])
fold_model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
check_model(fold_model)

print(f"Before folding: {len(fold_model.graph.node)} nodes")
for n in fold_model.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)})")

so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
outpath = os.path.join(TMPDIR, 'folded.onnx')
so.optimized_model_filepath = outpath
sess = ort.InferenceSession(fold_model.SerializeToString(), so)
folded = onnx.load(outpath)

print(f"\nAfter folding: {len(folded.graph.node)} nodes")
for n in folded.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)})")
print(f"\nFolded initializers:")
for init in folded.graph.initializer:
    arr = numpy_helper.to_array(init)
    print(f"  {init.name} = {arr}")

# Verify: f(100) = 100 - (2+3)*4 = 80
x_val = np.array([100.0], dtype=np.float32)
result = sess.run(None, {'x': x_val})[0]
expected_val = 100.0 - (2.0 + 3.0) * 4.0
print(f"\nf(100) = 100 - (2+3)*4 = {expected_val}")
print(f"Model output: {result[0]}")
assert np.isclose(result[0], expected_val)
print(f"Nodes eliminated: {len(fold_model.graph.node) - len(folded.graph.node)} (Add, Mul folded away)")

---
## Exercise 4: Conv + BatchNormalization Fusion

The most important optimization in CNN inference. BN parameters are absorbed into Conv weights:

$$W'_c = \frac{\gamma_c}{\sqrt{\sigma^2_c + \epsilon}} \cdot W_c, \quad
b'_c = \frac{\gamma_c(b_c - \mu_c)}{\sqrt{\sigma^2_c + \epsilon}} + \beta_c$$

This eliminates the BN kernel entirely — BN is **free** after fusion.

In [ ]:
np.random.seed(42)
C_in, C_out, kH, kW = 3, 16, 3, 3

W_conv = np.random.randn(C_out, C_in, kH, kW).astype(np.float32) * 0.1
b_conv = np.random.randn(C_out).astype(np.float32) * 0.01
gamma = np.ones(C_out, dtype=np.float32) * 0.9
beta  = np.zeros(C_out, dtype=np.float32)
mean  = np.random.randn(C_out).astype(np.float32) * 0.1
var   = np.abs(np.random.randn(C_out).astype(np.float32)) + 0.5
epsilon = 1e-5

inits = [
    numpy_helper.from_array(W_conv, 'W'), numpy_helper.from_array(b_conv, 'b'),
    numpy_helper.from_array(gamma, 'gamma'), numpy_helper.from_array(beta, 'beta'),
    numpy_helper.from_array(mean, 'mu'), numpy_helper.from_array(var, 'var'),
]

conv_node = helper.make_node('Conv', ['X', 'W', 'b'], ['h'],
                             kernel_shape=[3,3], pads=[1,1,1,1])
bn_node = helper.make_node('BatchNormalization',
                           ['h', 'gamma', 'beta', 'mu', 'var'], ['Y'])

X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, C_in, 32, 32])
Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, C_out, 32, 32])

graph = helper.make_graph([conv_node, bn_node], 'conv_bn',
                          inputs=[X_info], outputs=[Y_info], initializer=inits)
conv_bn_model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
check_model(conv_bn_model)

print(f"Before fusion: {len(conv_bn_model.graph.node)} nodes")
for n in conv_bn_model.graph.node:
    print(f"  {n.op_type}")

# Apply EXTENDED optimization (Conv+BN fusion lives here)
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
fused_path = os.path.join(TMPDIR, 'conv_bn_fused.onnx')
so.optimized_model_filepath = fused_path
sess_fused = ort.InferenceSession(conv_bn_model.SerializeToString(), so)
fused_model = onnx.load(fused_path)

print(f"\nAfter fusion: {len(fused_model.graph.node)} nodes")
for n in fused_model.graph.node:
    print(f"  {n.op_type}")

# Verify numerical equivalence
so_none = ort.SessionOptions()
so_none.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
sess_orig = ort.InferenceSession(conv_bn_model.SerializeToString(), so_none)

x_test = np.random.randn(1, C_in, 32, 32).astype(np.float32)
out_orig = sess_orig.run(None, {'X': x_test})[0]
out_fused = sess_fused.run(None, {'X': x_test})[0]

max_diff = np.max(np.abs(out_orig - out_fused))
print(f"\nNumerical verification:")
print(f"  Max |original - fused| = {max_diff:.2e}")
assert max_diff < 1e-4, f"Fusion changed outputs by {max_diff}"
print(f"  BN eliminated: {'BatchNormalization' not in [n.op_type for n in fused_model.graph.node]}")

---
## Exercise 5: MatMul + Add Fusion (→ Gemm)

A linear layer $y = Wx + b$ is exported as separate MatMul and Add nodes.
ORT fuses them into a single Gemm (General Matrix Multiply) kernel:

$$y = \alpha \cdot A B + \beta \cdot C \xrightarrow{\alpha=1,\beta=1} y = AB + C$$

**Memory savings** for $M{\times}K$ input, $K{\times}N$ weight:

$$\Delta T = 4MN \times \text{sizeof}(\text{float32}) \text{ bytes}$$

In [ ]:
M, K, N = 32, 256, 128

W_data = np.random.randn(K, N).astype(np.float32) * 0.02
b_data = np.random.randn(N).astype(np.float32) * 0.01

nodes = [
    helper.make_node('MatMul', ['X', 'W'], ['H']),
    helper.make_node('Add', ['H', 'b'], ['Y']),
]
inits = [
    numpy_helper.from_array(W_data, 'W'),
    numpy_helper.from_array(b_data, 'b'),
]

X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [M, K])
Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [M, N])

graph = helper.make_graph(nodes, 'matmul_add', [X_info], [Y_info], initializer=inits)
mm_model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
check_model(mm_model)

print(f"Before: {[n.op_type for n in mm_model.graph.node]}")

so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
outpath = os.path.join(TMPDIR, 'matmul_add_fused.onnx')
so.optimized_model_filepath = outpath
sess_fused = ort.InferenceSession(mm_model.SerializeToString(), so)
fused = onnx.load(outpath)

print(f"After:  {[n.op_type for n in fused.graph.node]}")
for n in fused.graph.node:
    for attr in n.attribute:
        print(f"  {n.op_type}.{attr.name} = {attr.f if attr.type==1 else attr.i if attr.type==2 else attr.s.decode() if attr.type==3 else '...'}")

# Verify + memory savings
so_none = ort.SessionOptions()
so_none.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
sess_orig = ort.InferenceSession(mm_model.SerializeToString(), so_none)

x_test = np.random.randn(M, K).astype(np.float32)
out_o = sess_orig.run(None, {'X': x_test})[0]
out_f = sess_fused.run(None, {'X': x_test})[0]
assert np.allclose(out_o, out_f, atol=1e-5)

savings_bytes = 4 * M * N * 4  # 4MN elements * 4 bytes
print(f"\nMemory traffic savings: {savings_bytes/1024:.1f} KB per inference")
print(f"  (eliminated 2 intermediate buffer read/writes of {M}x{N} tensors)")

---
## Exercise 6: Apply onnxoptimizer Passes

`onnxoptimizer` provides **offline** graph transformations that clean up ONNX files
before deployment. Unlike ORT (which optimizes at session creation), these produce
a portable `.onnx` file with no runtime dependency.

In [ ]:
if not HAS_OPTIMIZER:
    print("onnxoptimizer not installed — skipping this exercise.")
    print("Install with: pip install onnxoptimizer")
else:
    # Build a model with Conv+BN (fusible) and dead branches
    np.random.seed(0)
    C_in, C_out = 3, 8
    inits = [
        numpy_helper.from_array(np.random.randn(C_out, C_in, 3, 3).astype(np.float32)*0.1, 'W'),
        numpy_helper.from_array(np.zeros(C_out, dtype=np.float32), 'b'),
        numpy_helper.from_array(np.ones(C_out, dtype=np.float32), 'scale'),
        numpy_helper.from_array(np.zeros(C_out, dtype=np.float32), 'bias'),
        numpy_helper.from_array(np.zeros(C_out, dtype=np.float32), 'mean'),
        numpy_helper.from_array(np.ones(C_out, dtype=np.float32), 'var'),
    ]
    nodes = [
        helper.make_node('Conv', ['X', 'W', 'b'], ['c'], kernel_shape=[3,3], pads=[1,1,1,1]),
        helper.make_node('BatchNormalization', ['c', 'scale', 'bias', 'mean', 'var'], ['bn']),
        helper.make_node('Relu', ['bn'], ['Y']),
        helper.make_node('Sigmoid', ['c'], ['dead_sig']),  # dead branch
    ]
    X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, C_in, 16, 16])
    Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, C_out, 16, 16])
    g = helper.make_graph(nodes, 'optpass_demo', [X_i], [Y_i], initializer=inits)
    m = helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])
    check_model(m)

    print(f"Original: {len(m.graph.node)} nodes — {[n.op_type for n in m.graph.node]}")

    available = set(onnxoptimizer.get_available_passes())

    passes_to_try = [
        'eliminate_deadend',
        'eliminate_identity',
        'fuse_bn_into_conv',
        'fuse_consecutive_transposes',
    ]

    current = m
    print(f"\n{'Pass':<35} {'Nodes':>5}  Ops")
    print('-' * 70)
    print(f"{'(original)':<35} {len(current.graph.node):>5}  {[n.op_type for n in current.graph.node]}")
    for p in passes_to_try:
        if p not in available:
            print(f"{p:<35} {'N/A':>5}")
            continue
        try:
            current = onnxoptimizer.optimize(current, [p])
            print(f"{p:<35} {len(current.graph.node):>5}  {[n.op_type for n in current.graph.node]}")
        except Exception as e:
            print(f"{p:<35} error: {e}")

    check_model(current)
    print(f"\nFinal: {len(current.graph.node)} nodes (from {len(m.graph.node)})")

---
## Exercise 7: Performance Benchmark — Optimized vs Unoptimized

Build a multi-layer ConvNet and measure inference latency at each optimization level.
Graph optimization reduces latency by:
1. Eliminating kernel launches for removed nodes
2. Reducing memory traffic for fused operations
3. Enabling more efficient fused kernels

In [ ]:
def build_convnet(n_layers=3, C_in=3, base_channels=16, spatial=32):
    """Build a ConvNet with n_layers of Conv+BN+Relu + FC head."""
    np.random.seed(0)
    inits, nodes = [], []
    prev_c = C_in
    prev_name = 'X'

    for i in range(n_layers):
        c_out = base_channels * (2 ** i)
        Wname, bname = f'W{i}', f'b{i}'
        inits.append(numpy_helper.from_array(
            np.random.randn(c_out, prev_c, 3, 3).astype(np.float32)*0.1, Wname))
        inits.append(numpy_helper.from_array(
            np.zeros(c_out, dtype=np.float32), bname))
        for suffix, arr in [('s', np.ones(c_out)), ('bi', np.zeros(c_out)),
                            ('m', np.random.randn(c_out)*0.1), ('v', np.ones(c_out))]:
            inits.append(numpy_helper.from_array(arr.astype(np.float32), f'bn{i}_{suffix}'))

        cname = f'conv{i}'
        nodes.append(helper.make_node('Conv', [prev_name, Wname, bname], [cname],
                                      kernel_shape=[3,3], pads=[1,1,1,1]))
        nodes.append(helper.make_node('BatchNormalization',
            [cname, f'bn{i}_s', f'bn{i}_bi', f'bn{i}_m', f'bn{i}_v'], [f'bn{i}']))
        nodes.append(helper.make_node('Relu', [f'bn{i}'], [f'r{i}']))
        prev_c = c_out
        prev_name = f'r{i}'

    nodes.append(helper.make_node('GlobalAveragePool', [prev_name], ['gap']))
    nodes.append(helper.make_node('Flatten', ['gap'], ['flat'], axis=1))
    inits.append(numpy_helper.from_array(
        np.random.randn(prev_c, 10).astype(np.float32)*0.1, 'W_fc'))
    inits.append(numpy_helper.from_array(
        np.zeros(10, dtype=np.float32), 'b_fc'))
    nodes.append(helper.make_node('MatMul', ['flat', 'W_fc'], ['fc']))
    nodes.append(helper.make_node('Add', ['fc', 'b_fc'], ['Y']))

    X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, C_in, spatial, spatial])
    Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 10])
    g = helper.make_graph(nodes, 'convnet', [X_info], [Y_info], initializer=inits)
    return helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])

cnn = build_convnet(n_layers=4)
check_model(cnn)
print(f"Built ConvNet: {len(cnn.graph.node)} nodes")

x_bench = np.random.randn(1, 3, 32, 32).astype(np.float32)
n_warmup, n_runs = 100, 1000
latencies = {}

for name, level in levels:
    so = ort.SessionOptions()
    so.graph_optimization_level = level
    outpath = os.path.join(TMPDIR, f'bench_{name}.onnx')
    so.optimized_model_filepath = outpath
    sess = ort.InferenceSession(cnn.SerializeToString(), so,
                                providers=['CPUExecutionProvider'])
    for _ in range(n_warmup):
        sess.run(None, {'X': x_bench})

    t0 = time.perf_counter()
    for _ in range(n_runs):
        sess.run(None, {'X': x_bench})
    elapsed = (time.perf_counter() - t0) / n_runs * 1000

    opt_m = onnx.load(outpath)
    latencies[name] = elapsed
    print(f"  {name:<10}: {elapsed:.4f} ms  ({len(opt_m.graph.node)} nodes)")

baseline = latencies['DISABLED']
print(f"\nSpeedups relative to DISABLED:")
for name, lat in latencies.items():
    print(f"  {name:<10}: {baseline/lat:.2f}x")

In [ ]:
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    names = list(latencies.keys())
    times = list(latencies.values())
    colors = ['#FF6B6B', '#FFA500', '#4ECDC4', '#45B7D1']

    bars = ax1.bar(names, times, color=colors, edgecolor='black', linewidth=1.2)
    ax1.set_ylabel('Latency (ms)')
    ax1.set_title('Inference Latency by ORT Level', fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    for bar, t in zip(bars, times):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{t:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    speedups = [baseline/t for t in times]
    bars2 = ax2.bar(names, speedups, color=colors, edgecolor='black', linewidth=1.2)
    ax2.set_ylabel('Speedup (x)')
    ax2.set_title('Speedup Relative to DISABLED', fontweight='bold')
    ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    ax2.grid(axis='y', alpha=0.3)
    for bar, s in zip(bars2, speedups):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{s:.2f}x', ha='center', va='bottom', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not installed — skipping visualization.")

---
## Exercise 8: Numerical Equivalence Verification Protocol

Graph optimizations must preserve semantics. For float32, fusion may reorder
floating-point operations, introducing differences bounded by:

$$|\hat{y} - y| \leq n \cdot \mathbf{u} \cdot |y|, \quad \mathbf{u} = 2^{-24} \approx 6{\times}10^{-8}$$

We test with many random inputs and verify all differences are within tolerance.

In [ ]:
def verify_equivalence(model_bytes, input_shape, n_samples=200, tol=1e-5):
    """Run original vs fully-optimized and check numerical equivalence."""
    so_off = ort.SessionOptions()
    so_off.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    sess_off = ort.InferenceSession(model_bytes, so_off, providers=['CPUExecutionProvider'])

    so_on = ort.SessionOptions()
    so_on.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess_on = ort.InferenceSession(model_bytes, so_on, providers=['CPUExecutionProvider'])

    input_name = sess_off.get_inputs()[0].name
    max_diffs = []

    for _ in range(n_samples):
        x = np.random.randn(*input_shape).astype(np.float32)
        out_off = sess_off.run(None, {input_name: x})[0]
        out_on  = sess_on.run(None, {input_name: x})[0]
        max_diffs.append(np.max(np.abs(out_off - out_on)))

    max_diffs = np.array(max_diffs)
    return max_diffs

cnn = build_convnet(n_layers=3)
diffs = verify_equivalence(cnn.SerializeToString(), (1, 3, 32, 32), n_samples=300)

print("Equivalence Verification Report")
print("=" * 45)
print(f"  Samples tested:   {len(diffs)}")
print(f"  Max |diff|:       {diffs.max():.2e}")
print(f"  Mean |diff|:      {diffs.mean():.2e}")
print(f"  P99 |diff|:       {np.percentile(diffs, 99):.2e}")
print(f"  Tolerance:        1e-05")
print(f"  All within tol:   {np.all(diffs < 1e-5)}")

assert np.all(diffs < 1e-4), "Optimization changed model semantics!"
print("\nOptimization is numerically safe.")

---
## Challenge: Graph Optimization Benchmark Suite

Build a function that takes an arbitrary ONNX model, applies all four ORT optimization
levels, and produces a comprehensive report: node counts, file sizes, latency, and
numerical equivalence — all in one table.

In [ ]:
def optimization_benchmark(model, input_shape, n_warmup=50, n_runs=500):
    """Full optimization benchmark for an ONNX model."""
    model_bytes = model.SerializeToString()
    original_size = len(model_bytes)
    input_name = model.graph.input[0].name
    x_bench = np.random.randn(*input_shape).astype(np.float32)

    levels = [
        ('DISABLED',  ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
        ('BASIC',     ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
        ('EXTENDED',  ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
        ('ALL',       ort.GraphOptimizationLevel.ORT_ENABLE_ALL),
    ]

    results = []
    ref_output = None

    for name, level in levels:
        so = ort.SessionOptions()
        so.graph_optimization_level = level
        outpath = os.path.join(TMPDIR, f'suite_{name}.onnx')
        so.optimized_model_filepath = outpath
        sess = ort.InferenceSession(model_bytes, so, providers=['CPUExecutionProvider'])

        opt_model = onnx.load(outpath)
        opt_size = len(opt_model.SerializeToString())

        output = sess.run(None, {input_name: x_bench})[0]
        if ref_output is None:
            ref_output = output
        max_diff = np.max(np.abs(ref_output - output))

        for _ in range(n_warmup):
            sess.run(None, {input_name: x_bench})
        t0 = time.perf_counter()
        for _ in range(n_runs):
            sess.run(None, {input_name: x_bench})
        latency_ms = (time.perf_counter() - t0) / n_runs * 1000

        results.append({
            'level': name,
            'nodes': len(opt_model.graph.node),
            'size_kb': opt_size / 1024,
            'latency_ms': latency_ms,
            'max_diff': max_diff,
        })

    return results

# Run on our ConvNet
cnn = build_convnet(n_layers=4)
report = optimization_benchmark(cnn, (1, 3, 32, 32))

baseline_lat = report[0]['latency_ms']
print(f"{'Level':<12} {'Nodes':>5} {'Size(KB)':>9} {'Latency':>10} {'Speedup':>8} {'MaxDiff':>10}")
print('-' * 60)
for r in report:
    speedup = baseline_lat / r['latency_ms']
    print(f"{r['level']:<12} {r['nodes']:>5} {r['size_kb']:>9.1f} {r['latency_ms']:>8.4f}ms {speedup:>7.2f}x {r['max_diff']:>10.2e}")

print(f"\nOverall: {report[0]['nodes']} -> {report[-1]['nodes']} nodes "
      f"({100*(1-report[-1]['nodes']/report[0]['nodes']):.0f}% reduction)")

---
## Summary

| Concept | What You Practiced |
|:---|:---|
| Redundant ops | Built model with Identity, dead code; saw ORT eliminate them |
| ORT levels | DISABLED → BASIC → EXTENDED → ALL with node count comparison |
| Constant folding | Cascading fold of $c_1+c_2 \to c_{12}$, then $c_{12} \times c_3 \to c_{\text{final}}$ |
| Conv+BN fusion | $W' = \frac{\gamma}{\sqrt{\sigma^2+\epsilon}} W$, verified numerically |
| MatMul+Add → Gemm | Fusion saves $4MN \times$ sizeof bytes per inference |
| onnxoptimizer | Offline passes: `eliminate_deadend`, `fuse_bn_into_conv`, etc. |
| Performance | Measured real latency improvement across optimization levels |
| Verification | Confirmed $\|f_{\text{opt}} - f_{\text{orig}}\|_\infty < \epsilon$ over 300 samples |

**Next:** [Quantization Techniques](../02_Quantization_Techniques/)